# Stage A — Merged-file inventory & QA

Scan `/home/slow_data/Air_Quality/Stage_A/merged/YYYY/MM/DD/merged_YYYYMMDD_HHMM.nc` and check:

1. **Date / slot coverage** — which days are present, how many 30-min slots each day (max 48).
2. **Per-variable validity** — non-NaN pixel counts for every data variable.
3. **Sensor contribution** — distribution of `n_sensors`, `confidence_flag`, `dominant_sensor`.
4. **AOD value sanity** — min / max / mean / quantiles of `AOD_merged` and `AOD_phys_corrected`.
5. **Sample plot** — quick map of one slot.

Adjust `START` / `END` below to restrict the scan range.

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import re
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from config import MERGED_DIR, LATS, LONS, NLAT, NLON, SLOT_MINUTES

print('MERGED_DIR :', MERGED_DIR)
print('grid       :', NLAT, '×', NLON)
print('slot every :', SLOT_MINUTES, 'min  →', 24 * 60 // SLOT_MINUTES, 'slots/day')

## 1. Walk the directory tree

In [ ]:
# Restrict to a date range — set to None to scan everything.
START = None     # e.g. datetime(2024, 1, 1)
END   = None     # e.g. datetime(2024, 12, 31)

FNAME_RE = re.compile(r'^merged_(\d{8})_(\d{4})\.nc$')

rows = []
for p in Path(MERGED_DIR).rglob('merged_*.nc'):
    m = FNAME_RE.match(p.name)
    if not m:
        continue
    date_s, time_s = m.groups()
    ts = datetime.strptime(date_s + time_s, '%Y%m%d%H%M')
    if START and ts < START:
        continue
    if END and ts > END:
        continue
    rows.append({
        'path': p,
        'timestamp': ts,
        'date': ts.date(),
        'slot': ts.strftime('%H%M'),
        'size_kb': p.stat().st_size / 1024,
    })

inv = pd.DataFrame(rows).sort_values('timestamp').reset_index(drop=True)
print(f'Found {len(inv):,} merged NetCDF files.')
if len(inv):
    print(f'Range : {inv.timestamp.min()}  →  {inv.timestamp.max()}')
inv.head()

## 2. Daily slot coverage

Each calendar day should have 48 thirty-minute slots (00:00 → 23:30 UTC). Find days that fall short.

In [ ]:
SLOTS_PER_DAY = 24 * 60 // SLOT_MINUTES

daily = (
    inv.groupby('date')
       .agg(slots=('slot', 'nunique'), total_kb=('size_kb', 'sum'))
       .reset_index()
)
daily['missing'] = SLOTS_PER_DAY - daily['slots']

if len(daily):
    full_idx = pd.date_range(daily['date'].min(), daily['date'].max(), freq='D').date
    daily = daily.set_index('date').reindex(full_idx).reset_index()
    daily.rename(columns={'index': 'date'}, inplace=True)
    daily['slots'] = daily['slots'].fillna(0).astype(int)
    daily['missing'] = (SLOTS_PER_DAY - daily['slots']).astype(int)

print('Days with files          :', (daily['slots'] > 0).sum())
print('Days with 0 slots        :', (daily['slots'] == 0).sum())
print('Days fully covered (48)  :', (daily['slots'] == SLOTS_PER_DAY).sum())
print('Days partial (1..47)     :', ((daily['slots'] > 0) & (daily['slots'] < SLOTS_PER_DAY)).sum())

daily.head(10)

In [ ]:
# Days that are not fully covered
incomplete = daily[daily['slots'] != SLOTS_PER_DAY]
print(f'{len(incomplete)} days are not fully covered.')
incomplete.head(30)

In [ ]:
# Visualise daily slot coverage as a heatmap (rows=date, cols=slot)
if len(inv):
    pivot = (
        inv.assign(present=1)
           .pivot_table(index='date', columns='slot', values='present', fill_value=0)
    )
    full_slots = [f'{h:02d}{m:02d}' for h in range(24) for m in (0, 30)]
    pivot = pivot.reindex(columns=full_slots, fill_value=0)

    fig, ax = plt.subplots(figsize=(14, max(3, len(pivot) * 0.04)))
    ax.imshow(pivot.values, aspect='auto', cmap='Greys', interpolation='nearest')
    ax.set_xlabel('slot (HHMM UTC)')
    ax.set_ylabel('date')
    ax.set_title(f'Slot presence  ({pivot.values.sum():,} of {pivot.size:,} possible — '
                 f'{100*pivot.values.sum()/pivot.size:.1f}%)')
    step = max(1, len(full_slots) // 12)
    ax.set_xticks(range(0, len(full_slots), step))
    ax.set_xticklabels(full_slots[::step], rotation=45)
    step_y = max(1, len(pivot) // 20)
    ax.set_yticks(range(0, len(pivot), step_y))
    ax.set_yticklabels([str(d) for d in pivot.index[::step_y]])
    plt.tight_layout()
    plt.show()

## 3. Per-file content sweep

Sample (or scan-all) the inventory to record, per file:

- valid pixel count for each data variable
- AOD min / max / mean
- max `n_sensors`, dominant confidence flag

Scanning all 25k files is slow; the default takes one slot per day.

In [ ]:
# Sampling strategy: 'all'  | 'daily' (one slot/day) | int N (every Nth file)
SAMPLE = 'daily'

if SAMPLE == 'all':
    sample_df = inv
elif SAMPLE == 'daily':
    sample_df = inv.groupby('date', as_index=False).first()
elif isinstance(SAMPLE, int):
    sample_df = inv.iloc[::SAMPLE].copy()
else:
    raise ValueError(SAMPLE)

print(f'Scanning {len(sample_df):,} files (out of {len(inv):,}).')

In [ ]:
def summarise(path):
    """Return per-file diagnostics — robust to missing variables."""
    out = {'path': path}
    try:
        with xr.open_dataset(path) as ds:
            out['nlat'] = ds.sizes.get('lat')
            out['nlon'] = ds.sizes.get('lon')
            out['slot_utc'] = ds.attrs.get('slot_utc')
            out['vars'] = ';'.join(ds.data_vars)

            for v in ds.data_vars:
                arr = ds[v].values
                m = np.isfinite(arr)
                out[f'{v}__valid'] = int(m.sum())
                if v.startswith('AOD') and m.any():
                    sub = arr[m]
                    out[f'{v}__min']  = float(np.min(sub))
                    out[f'{v}__max']  = float(np.max(sub))
                    out[f'{v}__mean'] = float(np.mean(sub))
            if 'n_sensors' in ds.data_vars:
                ns = ds['n_sensors'].values
                if np.isfinite(ns).any():
                    out['n_sensors_max'] = float(np.nanmax(ns))
    except Exception as e:
        out['error'] = repr(e)
    return out

from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=8) as ex:
    results = list(ex.map(summarise, sample_df['path'].tolist()))

scan = pd.DataFrame(results)
scan['timestamp'] = pd.to_datetime(scan['slot_utc'])
print(f'Scanned {len(scan):,} files;  errors: {scan.get("error").notna().sum() if "error" in scan else 0}')
scan.head()

In [ ]:
# Files that failed to open
if 'error' in scan.columns:
    bad = scan[scan['error'].notna()]
    print(f'Files with read errors: {len(bad)}')
    display(bad[['path', 'error']].head(20))

In [ ]:
# Per-variable valid-pixel summary across scanned files
valid_cols = [c for c in scan.columns if c.endswith('__valid')]
if valid_cols:
    summary = pd.DataFrame({
        'mean_valid': scan[valid_cols].mean(),
        'min_valid' : scan[valid_cols].min(),
        'max_valid' : scan[valid_cols].max(),
        'pct_files_with_data': (scan[valid_cols] > 0).mean() * 100,
    })
    summary.index = [c.removesuffix('__valid') for c in summary.index]
    summary['mean_coverage_%'] = 100 * summary['mean_valid'] / (NLAT * NLON)
    summary = summary.sort_values('mean_valid', ascending=False)
    display(summary.round(2))

In [ ]:
# AOD value distribution across scanned files
for v in ['AOD_merged', 'AOD_phys_corrected']:
    cols = [f'{v}__min', f'{v}__max', f'{v}__mean']
    if all(c in scan.columns for c in cols):
        print(f'\n--- {v} ---')
        display(scan[cols].describe().round(3))

In [ ]:
# Time series: daily mean AOD_merged across scanned files
if 'AOD_merged__mean' in scan.columns:
    ts = scan.set_index('timestamp')['AOD_merged__mean'].dropna()
    if len(ts):
        daily_ts = ts.resample('D').mean()
        fig, ax = plt.subplots(figsize=(13, 3.5))
        ax.plot(daily_ts.index, daily_ts.values, lw=0.7)
        ax.set_ylabel('AOD_merged (daily mean)')
        ax.set_title('Domain-mean AOD_merged over time')
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

## 4. Sample slot — quick map

In [ ]:
if 'AOD_merged__mean' in scan.columns:
    sample_path = (
        scan.sort_values('AOD_merged__valid', ascending=False)
            .iloc[0]['path']
    )
else:
    sample_path = inv.iloc[len(inv) // 2]['path']

print('Plotting:', sample_path)
ds = xr.open_dataset(sample_path)
ds

In [ ]:
plot_vars = [v for v in ['AOD_merged', 'AOD_phys_corrected', 'n_sensors', 'confidence_flag']
             if v in ds.data_vars]

fig, axes = plt.subplots(1, len(plot_vars), figsize=(4.2 * len(plot_vars), 5),
                         constrained_layout=True)
if len(plot_vars) == 1:
    axes = [axes]
extent = (LONS[0] - 0.025, LONS[-1] + 0.025, LATS[-1] - 0.025, LATS[0] + 0.025)
for ax, v in zip(axes, plot_vars):
    arr = ds[v].values
    im = ax.imshow(arr, extent=extent, origin='upper',
                   cmap='viridis' if v.startswith('AOD') else 'plasma')
    ax.set_title(f'{v}\nvalid={np.isfinite(arr).sum():,}')
    ax.set_xlabel('lon'); ax.set_ylabel('lat')
    fig.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle(f"slot {ds.attrs.get('slot_utc')}", y=1.04)
plt.show()
ds.close()

## 5. Per-year / month rollup

In [ ]:
if len(inv):
    by_ym = (
        inv.assign(ym=inv['timestamp'].dt.to_period('M'))
           .groupby('ym')
           .agg(files=('path', 'size'),
                days=('date', 'nunique'),
                size_mb=('size_kb', lambda s: s.sum() / 1024))
           .reset_index()
    )
    by_ym['avg_slots_per_day'] = (by_ym['files'] / by_ym['days']).round(1)
    display(by_ym)